In [7]:
import pandas as pd

df = pd.read_csv("bank_transactions_data_2.csv")

print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())

  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   

  TransactionType   Location DeviceID      IP Address MerchantID Channel  \
0           Debit  San Diego  D000380  162.198.218.92       M015     ATM   
1           Debit    Houston  D000051     13.149.61.4       M052     ATM   
2           Debit       Mesa  D000235  215.97.143.157       M009  Online   
3           Debit    Raleigh  D000187  200.13.225.150       M002  Online   
4          Credit    Atlanta  D000308    65.164.3.100       M091  Online   

   CustomerAge CustomerOccupation  TransactionDuration  LoginAttempts  \
0           70             Doctor                   81 

### Step 1: Define Target Variable (`is_fraud`)

Since there's no explicit 'is_fraud' column, I'll create a synthetic one for demonstration. For this example, let's assume a transaction is 'fraudulent' if its `TransactionAmount` is unusually high (e.g., greater than 1000).

In [16]:
df['is_fraud'] = (df['TransactionAmount'] > 1000).astype(int)
print(df['is_fraud'].value_counts())

is_fraud
0    2422
1      90
Name: count, dtype: int64


### Step 2: Feature Selection and Data Preprocessing

I'll drop columns that are identifiers or date/time related as they might not be directly useful for a simple classification model or require more advanced feature engineering. Then, I will apply one-hot encoding to categorical features.

In [22]:
# Drop unnecessary columns for the model
# Keeping relevant columns for feature engineering or direct use
cols_to_drop = ['TransactionID', 'AccountID', 'DeviceID', 'IP Address', 'MerchantID', 'TransactionDate', 'PreviousTransactionDate']

# Drop columns, ignoring any that do not exist
df_processed = df.drop(columns=cols_to_drop, errors='ignore')

# Identify categorical columns for one-hot encoding
categorical_cols = df_processed.select_dtypes(include='object').columns

# Apply one-hot encoding
df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

# Display the first few rows of the processed DataFrame and its info
print(df_processed.head())
print(df_processed.info())

   TransactionAmount  CustomerAge  TransactionDuration  LoginAttempts  \
0              14.09           70                   81              1   
1             376.24           68                  141              1   
2             126.29           19                   56              1   
3             184.50           26                   25              1   
4              13.45           26                  198              1   

   AccountBalance  AccountID_AC00002  AccountID_AC00003  AccountID_AC00004  \
0         5112.21              False              False              False   
1        13758.91              False              False              False   
2         1122.35              False              False              False   
3         8569.06              False              False              False   
4         7429.40              False              False              False   

   AccountID_AC00005  AccountID_AC00006  ...  MerchantID_M098  \
0              False       

### Step 3: Split Data into Training and Testing Sets

Now, we'll separate our features (X) from our target variable (y) and then split them into training and testing sets to evaluate our model's performance on unseen data.

In [23]:
from sklearn.model_selection import train_test_split

X = df_processed.drop('is_fraud', axis=1)   # Features
y = df_processed['is_fraud']               # Target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y # stratify to maintain class distribution
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train value counts:\n{y_train.value_counts()}")
print(f"y_test value counts:\n{y_test.value_counts()}")

X_train shape: (2009, 4429)
X_test shape: (503, 4429)
y_train value counts:
is_fraud
0    1937
1      72
Name: count, dtype: int64
y_test value counts:
is_fraud
0    485
1     18
Name: count, dtype: int64


### Step 4: Model Training (Logistic Regression)

We'll use a Logistic Regression model as a baseline for this classification task. Before training, it's good practice to scale numerical features, but for simplicity, we'll proceed directly with training here. For more robust models, consider adding `StandardScaler`.

In [24]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model
# Set max_iter to a higher value if the model fails to converge
model = LogisticRegression(random_state=42, solver='liblinear', max_iter=200)
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


### Step 5: Model Evaluation

Finally, let's evaluate our model's performance on the test set using common classification metrics like accuracy, precision, recall, and F1-score.

In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9901
Precision: 0.9333
Recall: 0.7778
F1-Score: 0.8485

Confusion Matrix:
[[484   1]
 [  4  14]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       485
           1       0.93      0.78      0.85        18

    accuracy                           0.99       503
   macro avg       0.96      0.89      0.92       503
weighted avg       0.99      0.99      0.99       503



### Step 6: Save the Trained Model

To use the trained model later without needing to retrain it, we can save it to a file. The `pickle` module in Python is commonly used for this purpose, serializing the Python object structure into a byte stream.

In [26]:
import pickle

# Define the filename for the saved model
model_filename = 'logistic_regression_model.pkl'

# Save the trained model to a .pkl file
with open(model_filename, 'wb') as file:
    pickle.dump(model, file)

print(f"Model successfully saved to {model_filename}")

Model successfully saved to logistic_regression_model.pkl
